In [0]:
%python
# Ingest order data into the Bronze Delta table
source_path = "abfss://source@stecomdev0011.dfs.core.windows.net/orders/"
schema_path = "abfss://bronze@stecomdev0011.dfs.core.windows.net/_schemas/orders/"
checkpoint_path = "abfss://bronze@stecomdev0011.dfs.core.windows.net/_checkpoints/orders/"

df_orders = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

query_orders = (
    df_orders.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("ecom_dev.bronze.orders")
)

query_orders.awaitTermination()

In [0]:
%python
# Read and display data from the Bronze orders table
df_orders_bronze = spark.table("ecom_dev.bronze.orders")
display(df_orders_bronze)